# Домашнее задание по генерации речи

## Few-shot voice cloning модели XTTS


Cтатья: [XTTS: a Massively Multilingual Zero-Shot Text-to-Speech Model](https://arxiv.org/abs/2406.04904).

**github**: https://github.com/coqui-ai/TTS

**документация:** https://docs.coqui.ai/en/dev/models/xtts.html

**demo**: https://edresson.github.io/XTTS/

На семинаре были примеры спикеров, на которых zero-shot генерация по одной референсной записи показывала не очень высокий уровень похожести голоса. Альтернативным сценарием явялется дообучение модели на данных такого диктора, которое и предлагается выполнить в этом домашнем задании.

## 1. Настройка окружения

Установите необходимые пакеты, следуя документации (установка через pip либо скачивание репозитория с github).

## 2. Сбор и подготовка данных

2.1. подберите голос персонажа/актера/..., который плохо клонируется в zero-shot формате.

2.2. соберите как минимум 5-10 минут его голоса


In [1]:
import os
import io
import gc
import tempfile
import warnings
import logging
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr

import pandas as pd
import soundfile as sf
import shutil
import random

import torch
import torchaudio
import torch.nn.functional as F

from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector
from transformers.utils import logging as hf_logging
from TTS.api import TTS
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig, XttsAudioConfig
from TTS.utils.manage import ModelManager
from trainer import Trainer, TrainerArgs

from IPython.display import Audio, display

# Подавление warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*weights_only=False.*")
warnings.filterwarnings("ignore", message=".*mismatched key_padding_mask and attn_mask.*")

# Подавление логов от transformers/huggingface
hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("TTS").setLevel(logging.ERROR)

# Вспомогательная функция для тихого вызова TTS
def quiet_tts_to_file(tts_model, **kwargs):
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        tts_model.tts_to_file(**kwargs)

os.environ["COQUI_TOS_AGREED"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"

/home/anna/ssl_audio_env_py310_nemo/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/anna/ssl_audio_env_py310_nemo/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/anna/ssl_audio_env_py310_nemo/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/anna/ssl_audio_env_py310_nemo/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pyp

In [2]:
# Пути к данным
ROOT = Path("/home/anna/speach_recognition")

LIBRI_DIR = ROOT / "LibriSpeech"
RAW_DIR = ROOT / "data/raw_speaker"
OUT_DIR = DATASET_DIR = ROOT / "data/xtts_dataset"
WAVS_DIR = OUT_DIR / "wavs"

META_TRAIN = DATASET_DIR / "metadata_train.csv"
META_EVAL = DATASET_DIR / "metadata_eval.csv"
META_1MIN = DATASET_DIR / "metadata_train_1min.csv"

OUT_1MIN = OUT_PATH = ROOT / "xtts_1min_run"
OUT_FULL = ROOT / "xtts_full_run"
RESULTS = ROOT / "results_4_1"

CHECKPOINTS_OUT_PATH = OUT_PATH / "base_model_files"
OUT_WAVS = OUT_DIR / "wavs"

# Создаем необходимые директории
RAW_DIR.mkdir(parents=True, exist_ok=True)
WAVS_DIR.mkdir(parents=True, exist_ok=True)
OUT_1MIN.mkdir(parents=True, exist_ok=True)
OUT_FULL.mkdir(parents=True, exist_ok=True)
OUT_WAVS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_OUT_PATH.mkdir(parents=True, exist_ok=True)
# директории для результатов
(RESULTS / "zero_shot").mkdir(parents=True, exist_ok=True)
(RESULTS / "ft_1min").mkdir(parents=True, exist_ok=True)
(RESULTS / "ft_all").mkdir(parents=True, exist_ok=True)


In [ ]:
# начнем от обратного посмотрим какие спикеры имеют много аудиозаписей (по 5 минут и больше)
speaker2utt = {}  # speaker_id -> list[(audio_path, text, dur_sec)]

for trans_file in LIBRI_DIR.rglob("*.trans.txt"):
    for line in trans_file.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        utt_id, text = line.split(" ", 1)
        audio_path = trans_file.parent / f"{utt_id}.flac"
        if not audio_path.exists():
            continue

        info = sf.info(str(audio_path))
        dur = info.frames / info.samplerate
        speaker_id = utt_id.split("-")[0]
        speaker2utt.setdefault(speaker_id, []).append((audio_path, text, dur))

candidates = []
for spk, items in speaker2utt.items():
    total_sec = sum(x[2] for x in items)
    if total_sec >= 5 * 60:
        candidates.append((spk, total_sec, len(items)))

candidates = sorted(candidates, key=lambda x: x[1], reverse=True)

print("Кандидаты (top-10): speaker_id, minutes, num_utts")
for spk, sec, n in candidates[:10]:
    print(spk, round(sec / 60, 2), n)

Кандидаты (top-10): speaker_id, minutes, num_utts
4731 25.27 129
3274 25.26 123
2004 25.26 116
7540 25.26 128
188 25.26 113
7061 25.26 119
3851 25.26 126
1425 25.26 116
4057 25.26 112
7932 25.25 110


In [ ]:
# Теперь из 10 спикеров с большим количеством аудиозаписей посмотрим где получаются плохое симилярити для zero-shot
target_speakers = ["4731","3274","2004","7540","188","7061","3851","1425","4057","7932"]
sentences = [
    "The weather is nice today and this is a short cloning test.",
    "This is the same speaker for this sentence.",
    "This sentence checks voice consistency in zero-shot mode."
]

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
fe = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-sv")
sv = WavLMForXVector.from_pretrained("microsoft/wavlm-base-sv").to(device).eval()

def emb(p):
    w, sr = torchaudio.load(str(p))
    if w.shape[0] > 1: w = w.mean(0, keepdim=True)
    if sr != 16000: w = torchaudio.functional.resample(w, sr, 16000)
    x = fe(w.squeeze(0).numpy(), sampling_rate=16000, return_tensors="pt", padding=True)
    x = {k: v.to(device) for k, v in x.items()}
    with torch.inference_mode():
        return F.normalize(sv(**x).embeddings, dim=-1)[0]

rows = []
for spk in target_speakers:
    items = sorted(speaker2utt.get(spk, []), key=lambda x: x[0].name)
    if len(items) < 3:
        rows.append({"speaker_id": spk, "mean_similarity": None})
        continue

    refs = [items[i][0] for i in torch.linspace(0, len(items)-1, steps=3).long().tolist()]
    sims = []
    for ref in refs:
        r = emb(ref)
        for txt in sentences:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                p = Path(tmp.name)
            try:
                quiet_tts_to_file(tts, text=txt, file_path=str(p), speaker_wav=str(ref), language="en")
                sims.append(F.cosine_similarity(r.unsqueeze(0), emb(p).unsqueeze(0)).item())
            finally:
                if p.exists(): p.unlink()
    rows.append({"speaker_id": spk, "mean_similarity": sum(sims)/len(sims)})

df = pd.DataFrame(rows).sort_values("mean_similarity", ascending=True, na_position="last").reset_index(drop=True)
display(df)
print("Кандидат для дообучения:", df[df.mean_similarity.notna()].iloc[0]["speaker_id"])

 > Downloading model to /home/anna/snap/code/233/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2


100%|██████████| 1.87G/1.87G [08:34<00:00, 3.63MiB/s] 
4.37kiB [00:00, 6.06kiB/s]
361kiB [00:00, 615kiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 42.2iB/s]
 99%|█████████▉| 7.66M/7.75M [00:01<00:00, 4.14MiB/s]

 > Model's license - CPML
 > Check https://coqui.ai/cpml.txt for more info.
 > Using model: xtts


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/405M [00:00<?, ?B/s]

100%|██████████| 7.75M/7.75M [00:13<00:00, 4.14MiB/s]

,speaker_id,mean_similarity
0,7540,0.876122
1,7061,0.877270
2,3274,0.922782
3,188,0.925592
4,4731,0.941021
5,1425,0.941741
6,3851,0.947265
7,2004,0.957339
8,7932,0.963591
9,4057,0.964800


Кандидат для дообучения: 7540


In [ ]:
# я сначала попробовала спикера 7540 но почему-то при проверке оказалось что у него достаточно хороший показатель для зиро шот (больше 0,9) 
# по этому я взяла второго кандидата из списка
SELECTED_SPEAKER = "7061"  

items = sorted(speaker2utt[SELECTED_SPEAKER], key=lambda x: x[0].name)

total = 0.0
kept = 0
for i, (src_audio, text, dur) in enumerate(items):
    dst_audio = RAW_DIR / f"{SELECTED_SPEAKER}_{i:05d}.flac"
    dst_txt = dst_audio.with_suffix(".txt")

    shutil.copy2(src_audio, dst_audio)
    dst_txt.write_text(text.strip(), encoding="utf-8")

    total += dur
    kept += 1

print(f"Speaker: {SELECTED_SPEAKER}, files: {kept}, duration: {total/60:.2f} min")

Speaker: 7061, files: 119, duration: 25.26 min


In [ ]:
# 2.3. подготовьте данные в нужном формате для дообучения (можно использовать скрипты из репозитория)
################ ВАШ КОД
SPEAKER_NAME = "speaker7061"
EVAL_RATIO = 0.1
SEED = 42

audio_files = sorted(list(RAW_DIR.glob("*.flac")))

rows = []
skipped_no_text = 0
broken = 0

for i, ap in enumerate(audio_files):
    txt_path = ap.with_suffix(".txt")
    if not txt_path.exists():
        skipped_no_text += 1
        continue

    text = txt_path.read_text(encoding="utf-8").strip()
    if not text:
        skipped_no_text += 1
        continue

    try:
        audio, sr = sf.read(str(ap))
        out_name = f"{SPEAKER_NAME}_{i:05d}.wav"
        out_path = OUT_WAVS / out_name
        sf.write(str(out_path), audio, sr)

        rows.append({
            "audio_file": f"wavs/{out_name}",
            "text": text,
            "speaker_name": SPEAKER_NAME
        })
    except Exception:
        broken += 1

df = pd.DataFrame(rows).drop_duplicates(subset=["audio_file"]).reset_index(drop=True)

random.seed(SEED)
idx = list(df.index)
random.shuffle(idx)

eval_n = max(1, int(len(df) * EVAL_RATIO))
eval_idx = set(idx[:eval_n])

df_eval = df[df.index.isin(eval_idx)].reset_index(drop=True)
df_train = df[~df.index.isin(eval_idx)].reset_index(drop=True)

df_train.to_csv(OUT_DIR / "metadata_train.csv", sep="|", index=False, header=False)
df_eval.to_csv(OUT_DIR / "metadata_eval.csv", sep="|", index=False, header=False)

print(f"Train: {len(df_train)}, Eval: {len(df_eval)}, Total: {len(df)}")
print(f"Пропущено (нет текста): {skipped_no_text}, битых: {broken}")
print("Файлы: metadata_train.csv, metadata_eval.csv, папка wavs/")

Train: 108, Eval: 11, Total: 119
Пропущено (нет текста): 0, битых: 0
Файлы: metadata_train.csv, metadata_eval.csv, папка wavs/


In [36]:
# 2.4. Опишите полученный датасет

wav_files = sorted(WAVS_DIR.glob("*.wav"))

durations_sec = []
for p in wav_files:
    info = sf.info(str(p))
    durations_sec.append(info.frames / info.samplerate)

n_files = len(wav_files)
total_sec = sum(durations_sec)
avg_sec = total_sec / n_files if n_files > 0 else 0.0

print(f"Количество аудиозаписей: {n_files}")
print(f"Общая длительность: {total_sec/60:.2f} мин, средняя длительность записи: {avg_sec:.2f} сек")

Количество аудиозаписей: 119
Общая длительность: 25.26 мин, средняя длительность записи: 12.73 сек


## 3. Дообучение

3.1. в документации есть описание того, как происходит дообучение, прочтите необходимую информацию

3.2. Ответьте на вопрос: "Какая часть модели обновляется во время дообучения?"

################## ВАШ ОТВЕТ

При fine-tuning обновляется только **GPT-блок** — часть модели, которая выбирает акустические токены (звуки) на основе текста и голоса говорящего.

**Остальное заморожено:**
- **Вокодер (DVAE)** — превращает токены в аудио (не меняется)
- **Speaker Encoder** — извлекает особенности голоса (не меняется)
- **Нормализация спектрограмм** — техническая подготовка данных (не меняется)

**Почему это работает:**
- GPT содержит ~190М параметров — достаточно для адаптации к новому голосу
- Остальные части уже хорошо обучены на больших данных
- Всего 1-10 минут записей хватает, чтобы GPT понял особенности голоса


In [ ]:
# 3.3. Дообучите модель на 1 минуте данных

df_train_full = pd.read_csv(
            META_TRAIN,
            sep="|",
            header=None,
            names=["audio_file", "text", "speaker_name"],
        )

# 1 min subset 
picked = []
total_sec = 0.0
for _, r in df_train_full.iterrows():
    wav_path = DATASET_DIR / r["audio_file"]
    info = sf.info(str(wav_path))
    dur = info.frames / info.samplerate
    if total_sec >= 60.0:
        break
    picked.append(r)
    total_sec += dur

df_1min = pd.DataFrame(picked)
df_1min.to_csv(META_1MIN, sep="|", index=False, header=True)

dataset_config = BaseDatasetConfig(
    formatter="coqui",
    dataset_name="my_xtts_1min",
    path=str(DATASET_DIR),
    meta_file_train=META_1MIN.name,
    meta_file_val=META_EVAL.name,
    language="en",
)
DATASETS_CONFIG_LIST = [dataset_config]

# base checkpoints
DVAE_CHECKPOINT_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v1/v1.1.2/dvae.pth"
MEL_NORM_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v1/v1.1.2/mel_stats.pth"
TOKENIZER_FILE_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v1/v1.1.2/vocab.json"
XTTS_CHECKPOINT_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v1/v1.1.2/model.pth"

DVAE_CHECKPOINT = CHECKPOINTS_OUT_PATH / "dvae.pth"
MEL_NORM_FILE = CHECKPOINTS_OUT_PATH / "mel_stats.pth"
TOKENIZER_FILE = CHECKPOINTS_OUT_PATH / "vocab.json"
XTTS_CHECKPOINT = CHECKPOINTS_OUT_PATH / "model.pth"

to_download = []
for url, path in [
    (DVAE_CHECKPOINT_LINK, DVAE_CHECKPOINT),
    (MEL_NORM_LINK, MEL_NORM_FILE),
    (TOKENIZER_FILE_LINK, TOKENIZER_FILE),
    (XTTS_CHECKPOINT_LINK, XTTS_CHECKPOINT),
]:
    if not path.exists():
        to_download.append(url)

if to_download:
    ModelManager._download_model_files(to_download, str(CHECKPOINTS_OUT_PATH), progress_bar=True)

# training config
ref_rel = df_1min.iloc[0]["audio_file"]
ref_abs = str(DATASET_DIR / ref_rel)

model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    max_wav_length=255995,
    max_text_length=200,
    mel_norm_file=str(MEL_NORM_FILE),
    dvae_checkpoint=str(DVAE_CHECKPOINT),
    xtts_checkpoint=str(XTTS_CHECKPOINT),
    tokenizer_file=str(TOKENIZER_FILE),
    gpt_num_audio_tokens=8194,
    gpt_start_audio_token=8192,
    gpt_stop_audio_token=8193,
)

audio_config = XttsAudioConfig(
    sample_rate=22050,
    dvae_sample_rate=22050,
    output_sample_rate=24000,
)

config = GPTTrainerConfig(
    output_path=str(OUT_PATH),
    run_name="xtts_1min_ft",
    project_name="xtts_hw",
    dashboard_logger="tensorboard",
    model_args=model_args,
    audio=audio_config,
    batch_size=1,
    eval_batch_size=1,
    grad_clip=1.0,
    epochs=1,
    print_step=10,
    plot_step=100,
    save_step=200,
    save_n_checkpoints=1,
    save_checkpoints=True,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-6,
    num_loader_workers=2,
)

model_1 = GPTTrainer.init_from_config(config)

train_samples, eval_samples = load_tts_samples(
    DATASETS_CONFIG_LIST,
    eval_split=True,
)

# Очистка GPU памяти перед обучением
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=True,
        grad_accum_steps=16,
    ),
    config,
    output_path=str(OUT_PATH),
    model=model_1,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

trainer.fit()
print(f"Done. Checkpoints: {OUT_PATH}")

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 20
 | > Num. of Torch Threads: 1
 | > Torch seed: 1
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/home/anna/speach_recognition/xtts_1min_run/xtts_1min_ft-April-29-2026_09+49PM-6a2c2ab


>> DVAE weights restored from: /home/anna/speach_recognition/xtts_1min_run/base_model_files/dvae.pth
 | > Found 6 files in /home/anna/speach_recognition/data/xtts_dataset



 > Model has 514458563 parameters

 > EPOCH: 0/1
 --> /home/anna/speach_recognition/xtts_1min_run/xtts_1min_ft-April-29-2026_09+49PM-6a2c2ab

 > EVALUATION 



 > Filtering invalid eval samples!!
 > Total eval samples after filtering: 4



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.00751948356628418 (+0)
     | > avg_loss_text_ce: 0.025667833164334297 (+0)
     | > avg_loss_mel_ce: 3.3086355527242026 (+0)
     | > avg_loss: 3.334303379058838 (+0)



Done. Checkpoints: /home/anna/speach_recognition/xtts_1min_run


обновляется только языковая авторегрессионная часть модели (GPT-блок, который предсказывает акустические токены по тексту и референсу голоса).

Вокодер и остальные предобученные компоненты обычно заморожены и не обучаются заново.

Поэтому fine-tuning в этом ДЗ в первую очередь адаптирует модель под стиль/тембр конкретного диктора именно через GPT-часть.



In [ ]:
# 3.4. Дообучите модель на всех данных

# Используем базовые веса из 3.3
BASE = OUT_1MIN / "base_model_files"
DVAE_CHECKPOINT = BASE / "dvae.pth"
MEL_NORM_FILE = BASE / "mel_stats.pth"
TOKENIZER_FILE = BASE / "vocab.json"
XTTS_CHECKPOINT = BASE / "model.pth"

# model_args и audio_config уже определены в 3.3 - используем их
dataset_config = BaseDatasetConfig(
    formatter="coqui",
    dataset_name="my_xtts_full",
    path=str(DATASET_DIR),
    meta_file_train=META_TRAIN.name,
    meta_file_val=META_EVAL.name,
    language="en",
)
DATASETS_CONFIG_LIST = [dataset_config]

config = GPTTrainerConfig(
    output_path=str(OUT_PATH),
    run_name="xtts_full_ft",
    project_name="xtts_hw",
    dashboard_logger="tensorboard",
    model_args=model_args,
    audio=audio_config,
    batch_size=1,
    eval_batch_size=1,
    grad_clip=1.0,
    epochs=2,
    print_step=10,
    plot_step=100,
    save_step=200,
    save_n_checkpoints=1,
    save_checkpoints=True,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-6,
    num_loader_workers=2,
)

model_all = GPTTrainer.init_from_config(config)

train_samples, eval_samples = load_tts_samples(
    DATASETS_CONFIG_LIST,
    eval_split=True,
)

# Очистка GPU памяти перед обучением
import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=True,
        grad_accum_steps=12,
    ),
    config,
    output_path=str(OUT_PATH),
    model=model_all,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

trainer.fit()
print(f"Done. Checkpoints: {OUT_PATH}")

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 20
 | > Num. of Torch Threads: 1
 | > Torch seed: 1
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/home/anna/speach_recognition/xtts_full_run/xtts_full_ft-April-29-2026_09+47PM-6a2c2ab


>> DVAE weights restored from: /home/anna/speach_recognition/xtts_1min_run/base_model_files/dvae.pth
 | > Found 108 files in /home/anna/speach_recognition/data/xtts_dataset



 > Model has 518128803 parameters

 > EPOCH: 0/2
 --> /home/anna/speach_recognition/xtts_full_run/xtts_full_ft-April-29-2026_09+47PM-6a2c2ab

 > EVALUATION 



 > Filtering invalid eval samples!!
 > Total eval samples after filtering: 4



  --> EVAL PERFORMANCE
     | > avg_loader_time: 0.010849952697753906 (+0)
     | > avg_loss_text_ce: 0.025667833164334297 (+0)
     | > avg_loss_mel_ce: 3.3086355527242026 (+0)
     | > avg_loss: 3.334303379058838 (+0)


 > EPOCH: 1/2
 --> /home/anna/speach_recognition/xtts_full_run/xtts_full_ft-April-29-2026_09+47PM-6a2c2ab

 > TRAINING (2026-04-29 21:47:34) 


...

[!] Warning: The text length exceeds the character limit of 250 for language 'en', this might cause truncated audio.
[!] Warning: The text length exceeds the character limit of 250 for language 'en', this might cause truncated audio.



   --> TIME: 2026-04-29 21:47:43 -- STEP: 40/108 -- GLOBAL_STEP: 40
     | > loss_text_ce: 0.031709395349025726  (0.025390183785930276)
     | > loss_mel_ce: 4.667673587799072  (5.09857132434845)
     | > loss: 0.39161524176597595  (0.42699680253863337)
     | > current_lr: 5e-06 
     | > step_time: 0.0933  (0.15195514559745787)
     | > loader_time: 0.0028  (0.0045405149459838865)



[!] Warning: The text length exceeds the character limit of 250 for language 'en', this might cause truncated audio.



   --> TIME: 2026-04-29 21:47:45 -- STEP: 50/108 -- GLOBAL_STEP: 50
     | > loss_text_ce: 0.0262795090675354  (0.02558621618896723)
     | > loss_mel_ce: 5.164480209350586  (5.076810531616211)
     | > loss: 0.43256330490112305  (0.42519973874092104)
     | > current_lr: 5e-06 
     | > step_time: 0.1244  (0.15547001361846924)
     | > loader_time: 0.0025  (0.0044236421585083)


   --> TIME: 2026-04-29 21:47:47 -- STEP: 60/108 -- GLOBAL_STEP: 60
     | > loss_text_ce: 0.026554210111498833  (0.025253540929406883)
     | > loss_mel_ce: 5.038445472717285  (5.014078958829244)
     | > loss: 0.42208331823349  (0.41994438469409945)
     | > current_lr: 5e-06 
     | > step_time: 0.1025  (0.1571067969004313)
     | > loader_time: 0.0025  (0.004114317893981933)


   --> TIME: 2026-04-29 21:47:49 -- STEP: 70/108 -- GLOBAL_STEP: 70
     | > loss_text_ce: 0.021031003445386887  (0.025155877242130892)
     | > loss_mel_ce: 4.517190456390381  (4.9743566308702745)
     | > loss: 0.37818512320518494

Done. Checkpoints: /home/anna/speach_recognition/xtts_full_run


## 4. Оценка результатов

**4.1.** Для каждой из 3х моделей (zero-shot, дообученная на 1 минуте, дообученная на всех данных) сгенерируйте 10 предложени


In [ ]:
# 4.1. Генерация 10 предложений для 3 моделей

all_wavs = sorted(WAVS_DIR.glob("*.wav"))

# Первые 3 зарезервированы для оценки similarity (ref_paths в 4.2)
# Для кондиционирования берем 4-й файл, чтобы не пересекаться с референсами оценки
REF_WAV = all_wavs[3] if len(all_wavs) > 3 else all_wavs[-1]
print(f"Referencing WAV for generation: {REF_WAV.name}")


SENTENCES = [
    "The quick brown fox jumps over the lazy dog.",
    "This voice should sound similar to the reference speaker.",
    "Fine tuning can improve timbre consistency.",
    "Today we evaluate three XTTS models on the same text.",
    "Small datasets may lead to unstable pronunciation.",
    "Longer adaptation usually improves speaker similarity.",
    "Please compare clarity, prosody, and naturalness.",
    "This is a sample sentence for qualitative listening.",
    "The model should preserve identity across different prompts.",
    "End of the generation benchmark.",
]

# 1) zero-shot (базовая XTTS)
tts_zero = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
for i, text in enumerate(SENTENCES, 1):
    quiet_tts_to_file(
        tts_zero,
        text=text,
        file_path=str(RESULTS / "zero_shot" / f"{i:02d}.wav"),
        speaker_wav=str(REF_WAV),
        language="en",
    )

# 2) fine-tuned 1 min (из model_1)
xtts_1 = model_1.xtts
xtts_1.to(device)
xtts_1.eval()

gpt_cond_1, spk_emb_1 = xtts_1.get_conditioning_latents(audio_path=[str(REF_WAV)])
for i, text in enumerate(SENTENCES, 1):
    out = xtts_1.inference(
        text=text,
        language="en",
        gpt_cond_latent=gpt_cond_1,
        speaker_embedding=spk_emb_1,
        temperature=0.7,
    )
    sf.write(str(RESULTS / "ft_1min" / f"{i:02d}.wav"), out["wav"], 24000)

# 3) fine-tuned all (из model_all)
xtts_all = model_all.xtts
xtts_all.to(device)
xtts_all.eval()

gpt_cond_all, spk_emb_all = xtts_all.get_conditioning_latents(audio_path=[str(REF_WAV)])
for i, text in enumerate(SENTENCES, 1):
    out = xtts_all.inference(
        text=text,
        language="en",
        gpt_cond_latent=gpt_cond_all,
        speaker_embedding=spk_emb_all,
        temperature=0.7,
    )
    sf.write(str(RESULTS / "ft_all" / f"{i:02d}.wav"), out["wav"], 24000)

Referencing WAV for generation: speaker7061_00003.wav
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


In [ ]:
# 4.1. Отображение примеров
# GT - реальные аудио этого спикера 
all_wavs = sorted(WAVS_DIR.glob("*.wav"))
gt = all_wavs[3] if len(all_wavs) > 3 else all_wavs[-1]

zero = sorted((RESULTS / "zero_shot").glob("*.wav"))[0]
ft1 = sorted((RESULTS / "ft_1min").glob("*.wav"))[0]
ftall = sorted((RESULTS / "ft_all").glob("*.wav"))[0]

print("Ground Truth (референс для кондиционирования):")
display(Audio(filename=str(gt)))

print("\nZero-shot:")
display(Audio(filename=str(zero)))

print("\nFew-shot (1 min):")
display(Audio(filename=str(ft1)))

print("\nFew-shot (all data):")
display(Audio(filename=str(ftall)))


Ground Truth (референс для кондиционирования):



Zero-shot:



Few-shot (1 min):



Few-shot (all data):


**4.2.** Для каждой из моделей оцените speaker similarity моделью [WavLM-sv](https://huggingface.co/microsoft/wavlm-base-sv), как мы делали на семинаре

!!! в качестве референсной записи выберите 3 различных варианта и подсчитайте similarity:

- этих записей друг с другом

- этих записей со всеми сгенерированными

In [ ]:
# 4.2 Speaker similarity через WavLM-sv

# Берем 3 разных референса GT
# Считаем similarity референсов друг с другом
# Считаем similarity референсов со всеми сгенерированными для каждой модели

ref_paths = sorted(WAVS_DIR.glob("*.wav"))[:3]
zero_paths = sorted((RESULTS / "zero_shot").glob("*.wav"))
ft1_paths = sorted((RESULTS / "ft_1min").glob("*.wav"))
ftall_paths = sorted((RESULTS / "ft_all").glob("*.wav"))

def cos(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# Эмбеддинги референсов (переиспользуем emb)
ref_embs = [emb(p) for p in ref_paths]

# 1) Similarity референсов друг с другом
ref_pair_sims = []
for i in range(len(ref_embs)):
    for j in range(i + 1, len(ref_embs)):
        ref_pair_sims.append(cos(ref_embs[i], ref_embs[j]))
speaker_similarity_gt = sum(ref_pair_sims) / len(ref_pair_sims)

# 2) Similarity референсов со всеми сгенерированными
def mean_ref_to_generated(gen_paths):
    gen_embs = [emb(p) for p in gen_paths]
    sims = [cos(r, g) for r in ref_embs for g in gen_embs]
    return sum(sims) / len(sims)

speaker_similarity_zero = mean_ref_to_generated(zero_paths)
speaker_similarity_1min = mean_ref_to_generated(ft1_paths)
speaker_similarity_full = mean_ref_to_generated(ftall_paths)

print(f"Speaker similarity GT (ground truth примеры между собой): {speaker_similarity_gt:.4f}")
print(f"Speaker similarity zero-shot: {speaker_similarity_zero:.4f}")
print(f"Speaker similarity few-shot (1 мин): {speaker_similarity_1min:.4f}")
print(f"Speaker similarity few-shot (все данные): {speaker_similarity_full:.4f}")

Speaker similarity GT (ground truth примеры между собой): 0.9277
Speaker similarity zero-shot: 0.8760
Speaker similarity few-shot (1 мин): 0.9014
Speaker similarity few-shot (все данные): 0.9079


В целом очень интересно как всего с 1 минутой данных можно получить настолько хороший результат. Странно что при обучении на 25 минутах результат не получился намного ближе к оригиналу, хотя небольшой прирост присутствует
